In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

26/07/01 05:02:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [22]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("iceberg-query")
    # Iceberg Spark
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )

    # Register the default catalog
    .config(
        "spark.sql.catalog.default",
        "org.apache.iceberg.spark.SparkCatalog",
    )
    .config("spark.sql.catalog.default.type", "rest")
    .config("spark.sql.catalog.default.uri", "http://rest:8181")
    .config("spark.sql.catalog.default.warehouse", "s3://warehouse/")
    .config(
        "spark.sql.catalog.default.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO",
    )
    .config("spark.sql.catalog.default.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.default.default-namespace", "silver")

    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    # Make 'default' the current catalog
    .config("spark.sql.defaultCatalog", "default")

    .getOrCreate()
)

26/07/01 06:41:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [21]:
spark.stop()

In [23]:
spark.sql("""
ALTER TABLE silver.penelitian
SET TBLPROPERTIES (
    'write.wap.enabled'='true'
)
""").toPandas()

""


In [24]:
spark.sql("select 1;")

DataFrame[1: int]

In [67]:
commits = "delete_ditolak"

In [27]:
# Normal data

In [68]:
spark.sql("SELECT status, COUNT(*) AS cnt FROM silver.penelitian GROUP BY status").show()

+---------+---+
|   status|cnt|
+---------+---+
|  ditolak|511|
| diterima|391|
|diusulkan| 29|
+---------+---+



In [69]:
spark.sql(f"ALTER TABLE silver.penelitian CREATE BRANCH IF NOT EXISTS wap_{commits}")

DataFrame[]

In [70]:
spark.conf.set('spark.wap.branch', f"wap_{commits}")

In [72]:
spark.sql("""
delete from default.silver.penelitian where status = 'ditolak'
""")

DataFrame[]

In [74]:
print("STAGED (branch):")
spark.sql(f"SELECT status, COUNT(*) AS cnt FROM default.silver.penelitian VERSION AS OF 'wap_{commits}' GROUP BY status").show()

print("PUBLISHED (main):")
spark.sql("SELECT status, COUNT(*) AS cnt FROM default.silver.penelitian VERSION AS OF 'main' GROUP BY status").show()

STAGED (branch):
+---------+---+
|   status|cnt|
+---------+---+
| diterima|391|
|diusulkan| 29|
+---------+---+

PUBLISHED (main):
+---------+---+
|   status|cnt|
+---------+---+
|  ditolak|511|
| diterima|391|
|diusulkan| 29|
+---------+---+



In [66]:
# spark.sql(f"""
# CALL default.system.cherrypick_snapshot('silver.penelitian','wap_{commits}')
# """)

In [75]:
spark.sql(f"""
CALL default.system.fast_forward(
    'silver.penelitian',
    'main',
    'wap_{commits}'
)
""").toPandas()

,branch_updated,previous_ref,updated_ref
0,main,5261379504122657845,2539675533103788183


In [76]:
spark.sql("select distinct status from default.silver.penelitian").toPandas()

,status
0,diterima
1,diusulkan
